# 02 — CORE Data Exploration

This notebook documents the raw CORE social lettings microdata before any pipeline work. It covers the schema challenge (two incompatible file formats across years), key column distributions, and the data quality issues that the ingestion step had to handle.

**Source**: UK Data Service, SN 9237 — CORE (Continuous Recording of Letting and Sales in Social Housing), 2007–2022.

**A row represents**: One new social housing letting — the moment a household moves into a social rented property. Each letting has details about the property, the household, income, rent, previous situation, and reason for letting.

In [1]:
import os, glob
os.environ['JAVA_HOME'] = '/opt/homebrew/opt/openjdk@17'
os.environ['PYSPARK_PYTHON'] = '/Users/user/Documents/repos/.venv/bin/python'
os.environ['PYSPARK_DRIVER_PYTHON'] = '/Users/user/Documents/repos/.venv/bin/python'
os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, avg, min, max, trim, when, round as spark_round

spark = SparkSession.builder \
    .master('local[*]') \
    .appName('explore_core') \
    .config('spark.driver.memory', '4g') \
    .getOrCreate()
spark.sparkContext.setLogLevel('ERROR')

BRONZE = '../data/bronze/core_raw/tab'
all_files = sorted(glob.glob(f'{BRONZE}/*.tab'))
print(f'Total tab files: {len(all_files)}')
for f in all_files:
    print(' ', os.path.basename(f))

Total tab files: 42
  0708_sr_gn_eul.tab
  0708_sr_sh_eul.tab
  0809_sr_gn_eul.tab
  0809_sr_sh_eul.tab
  0910_sr_gn_eul.tab
  0910_sr_sh_eul.tab
  1011_sr_gn_eul.tab
  1011_sr_sh_eul.tab
  1112_ar_gn_eul.tab
  1112_ar_sh_eul.tab
  1112_sr_gn_eul.tab
  1112_sr_sh_eul.tab
  1213_ar_gn_eul.tab
  1213_ar_sh_eul.tab
  1213_sr_gn_eul.tab
  1213_sr_sh_eul.tab
  1314_ar_gn_eul.tab
  1314_ar_sh_eul.tab
  1314_sr_gn_eul.tab
  1314_sr_sh_eul.tab
  1415_ar_gn_eul.tab
  1415_ar_sh_eul.tab
  1415_sr_gn_eul.tab
  1415_sr_sh_eul.tab
  1516_ar_gn_eul.tab
  1516_ar_sh_eul.tab
  1516_sr_gn_eul.tab
  1516_sr_sh_eul.tab
  1617_ar_gn_eul.tab
  1617_ar_sh_eul.tab
  1617_sr_gn_eul.tab
  1617_sr_sh_eul.tab
  1718_ar_gn_eul.tab
  1718_ar_sh_eul.tab
  1718_rb_gn_eul.tab
  1718_rb_sh_eul.tab
  1718_sr_gn_eul.tab
  1718_sr_sh_eul.tab
  core_lettings_2018-19_eul.tab
  core_lettings_2019-20_eul.tab
  core_lettings_2020-21_eul.tab
  core_lettings_2021-22_eul.tab


## 1. The schema problem — two file formats

This is the most important discovery in exploration. The 42 files do not share a common schema.

In [2]:
# Count columns in each file to reveal schema groups
print(f'{"File":<45} {"Columns":>8}')
print('-' * 55)
for path in all_files:
    name = os.path.basename(path)
    df = spark.read.csv(path, header=True, inferSchema=False, sep='\t')
    print(f'{name:<45} {len(df.columns):>8}')

File                                           Columns
-------------------------------------------------------


0708_sr_gn_eul.tab                                  99
0708_sr_sh_eul.tab                                  97
0809_sr_gn_eul.tab                                  99


0809_sr_sh_eul.tab                                  99
0910_sr_gn_eul.tab                                 104
0910_sr_sh_eul.tab                                 104


1011_sr_gn_eul.tab                                 103
1011_sr_sh_eul.tab                                 104
1112_ar_gn_eul.tab                                  93


1112_ar_sh_eul.tab                                  94
1112_sr_gn_eul.tab                                 102
1112_sr_sh_eul.tab                                 103
1213_ar_gn_eul.tab                                  98


...

1516_ar_gn_eul.tab                                 113
1516_ar_sh_eul.tab                                 112
1516_sr_gn_eul.tab                                 124
1516_sr_sh_eul.tab                                 123


1617_ar_gn_eul.tab                                 112
1617_ar_sh_eul.tab                                 111
1617_sr_gn_eul.tab                                 123
1617_sr_sh_eul.tab                                 122


1718_ar_gn_eul.tab                                 117
1718_ar_sh_eul.tab                                 116
1718_rb_gn_eul.tab                                 117
1718_rb_sh_eul.tab                                 116
1718_sr_gn_eul.tab                                 117


1718_sr_sh_eul.tab                                 116
core_lettings_2018-19_eul.tab                      203
core_lettings_2019-20_eul.tab                      205
core_lettings_2020-21_eul.tab                      207


core_lettings_2021-22_eul.tab                      213


The column count ranges from 93 to 213. This means **globbing all files together will cause column misalignment** — Spark would use one file's header for all files, putting data into the wrong columns for every other schema variant.

Solution (implemented in `04_ingest_core.ipynb`): read each file individually in a loop, select columns by name (not position), then union.

## 2. Geography — how the UK is divided and how CORE uses it

### UK geography hierarchy

The UK uses a standard hierarchy of geography codes (ONS codes) from largest to smallest:

| Level | Example | Code prefix | Count in England |
|---|---|---|---|
| Country | England | E92 | 1 |
| Government Office Region (GOR) | London | E12 | 9 |
| Local Authority / Borough | Hackney | E09 (London), E06/E07/E08 elsewhere | 33 in London |
| LSOA (Lower Super Output Area) | ~1,500 households | E01 | ~4,835 in London |
| OA (Output Area) | ~300 households | E00 | smallest census unit |

**The 9 Government Office Regions in England:**

| Code | Region |
|---|---|
| E12000001 | North East |
| E12000002 | North West |
| E12000003 | Yorkshire and The Humber |
| E12000004 | East Midlands |
| E12000005 | West Midlands |
| E12000006 | East of England |
| **E12000007** | **London** ← what we filter on |
| E12000008 | South East |
| E12000009 | South West |

Scotland, Wales and Northern Ireland use W, S, N prefixes and are outside the E12 system.

### How CORE uses geography

CORE anonymises data to **region level only** in the public end-user licence version. This means:
- We can identify that a letting is in London (`GOVREG = E12000007`) ✓
- We **cannot** identify which London borough it is in ✗

Borough-level financial vulnerability therefore comes from a separate source — IMD 2019, which is published at LSOA level and tagged with the parent Local Authority (E09 = London borough).

Additionally, **old CORE files (2007–2018) use a numeric shorthand** — `GOVREG = '7'` instead of `'E12000007'`. Our filter handles both:
```python
.filter((trim(col('GOVREG')) == '7') | (trim(col('GOVREG')) == 'E12000007'))
```

In [3]:
# Check GOVREG values in old vs new files
old = spark.read.csv(f'{BRONZE}/0708_sr_gn_eul.tab', header=True, inferSchema=False, sep='\t')
new = spark.read.csv(f'{BRONZE}/core_lettings_2018-19_eul.tab', header=True, inferSchema=False, sep='\t')

print('=== GOVREG values in old file (0708) ===')
display(old.groupBy('GOVREG').count().orderBy('GOVREG').toPandas().style.format(thousands=","))

print('=== GOVREG values in new file (2018-19) ===')
display(new.groupBy(trim(col('GOVREG')).alias('GOVREG')).count().orderBy('GOVREG').toPandas().style.format(thousands=","))

=== GOVREG values in old file (0708) ===


,GOVREG,count
0,1,"17,412"
1,2,"23,162"
2,3,"16,935"
3,4,"19,915"
4,5,"18,779"
5,6,"23,265"
6,7,"17,312"
7,8,"26,692"
8,9,"35,453"


=== GOVREG values in new file (2018-19) ===


,GOVREG,count
0,,623
1,E12000001,"25,773"
2,E12000002,"50,702"
3,E12000003,"33,636"
4,E12000004,"24,646"
5,E12000005,"39,289"
6,E12000006,"26,854"
7,E12000007,"26,669"
8,E12000008,"31,448"
9,E12000009,"27,764"


In [4]:
# Confirm: GOVREG=7 in old files = London = E12000007 in new files
print('Old file, GOVREG=7 row count:', old.filter(col('GOVREG') == '7').count())
print('New file, GOVREG=E12000007 row count:', new.filter(trim(col('GOVREG')) == 'E12000007').count())
print('\nNote: CORE geography is region-level only — no borough breakdown available in this dataset.')

Old file, GOVREG=7 row count: 17312


New file, GOVREG=E12000007 row count: 26669

Note: CORE geography is region-level only — no borough breakdown available in this dataset.


## 3. Income and rent bands — what do they look like raw?

Income and rent are not stored as numbers — they are string bands. Understanding the format is needed to build the midpoint UDF.

In [5]:
london_new = new.filter(trim(col('GOVREG')) == 'E12000007')

print('=== Weekly income bands (sample) ===')
display(london_new.groupBy('WEEKINC_T_Bands').count().orderBy('WEEKINC_T_Bands').limit(20).toPandas().style.format(thousands=","))

print('=== Weekly rent bands (sample) ===')
display(london_new.groupBy('WRENT_Bands').count().orderBy('WRENT_Bands').limit(20).toPandas().style.format(thousands=","))

=== Weekly income bands (sample) ===


,WEEKINC_T_Bands,count
0,,"17,688"
1,120.00 to 149.99,744
2,150.00 to 189.99,"1,025"
3,190.00 to 219.99,620
4,220.00 to 269.99,"1,106"
5,270.00 to 319.99,942
6,320.00 to 369.99,831
7,370.00 to 419.99,659
8,420.00 to 469.99,547
9,470.00 to 519.99,422


=== Weekly rent bands (sample) ===


,WRENT_Bands,count
0,,107
1,110.00 to 129.99,"6,565"
2,130.00 to 149.99,"3,118"
3,150.00 to 169.99,"1,811"
4,170.00 to 199.99,991
5,200.00 or more,"2,411"
6,60.00 to 64.99,25
7,65.00 to 69.99,147
8,70.00 to 74.99,209
9,75.00 to 79.99,552


## 4. Blank string issue in integer columns

Several numeric columns contain blank strings instead of nulls. A direct `.cast('int')` will fail — this must be handled before casting.

In [6]:
int_cols = ['HHMEMBT', 'BEDST', 'BED_MINUS_BEDSTANDARD', 'YEAR']
london_new_cached = london_new.cache()
total = london_new_cached.count()

print(f'London rows in 2018-19 file: {total:,}\n')
print(f'{"Column":<30} {"Blank strings":>15} {"Blank %":>8}')
print('-' * 55)
for c in int_cols:
    if c in london_new.columns:
        blanks = london_new_cached.filter(trim(col(c)) == '').count()
        print(f'{c:<30} {blanks:>15,} {blanks/total*100:>7.1f}%')

London rows in 2018-19 file: 26,669

Column                           Blank strings  Blank %
-------------------------------------------------------
HHMEMBT                                      0     0.0%
BEDST                                    9,045    33.9%
BED_MINUS_BEDSTANDARD                   10,245    38.4%


YEAR                                         0     0.0%


## 5. London rows across all years — using the correct filter

In [7]:
# Read each file individually and count London rows
print(f'{"File":<45} {"London rows":>12}')
print('-' * 59)
total_london = 0
for path in all_files:
    name = os.path.basename(path)
    df = spark.read.csv(path, header=True, inferSchema=False, sep='\t')
    n = df.filter(
        (trim(col('GOVREG')) == '7') | (trim(col('GOVREG')) == 'E12000007')
    ).count()
    total_london += n
    print(f'{name:<45} {n:>12,}')
print('-' * 59)
print(f'{"TOTAL":<45} {total_london:>12,}')

File                                           London rows
-----------------------------------------------------------
0708_sr_gn_eul.tab                                  17,312


0708_sr_sh_eul.tab                                  11,871


0809_sr_gn_eul.tab                                  18,741
0809_sr_sh_eul.tab                                  11,439


0910_sr_gn_eul.tab                                  15,546
0910_sr_sh_eul.tab                                  10,088


1011_sr_gn_eul.tab                                  18,035
1011_sr_sh_eul.tab                                  13,593


...

1718_sr_gn_eul.tab                                  14,354
1718_sr_sh_eul.tab                                   8,107


core_lettings_2018-19_eul.tab                       26,669


core_lettings_2019-20_eul.tab                       28,303


core_lettings_2020-21_eul.tab                       21,473


core_lettings_2021-22_eul.tab                       23,505
-----------------------------------------------------------
TOTAL                                              501,244


## 6. Key column availability across schema versions

Not all columns exist in all years. These are the columns we want and which years they appear in.

In [8]:
target_cols = [
    'YEAR', 'GOVREG', 'LETTYPE', 'TENANCY', 'HHMEMBT', 'BEDST',
    'BED_MINUS_BEDSTANDARD', 'BED_MINUS_BEDSTANDARD2',
    'PREVTEN_R', 'REASON_R', 'WEEKINC_T_Bands', 'WRENT_Bands',
    'WTSHORTFALLHB_Bands', 'WTSHORTFALL_Bands',
    'ETHNIC_Bands', 'TENANCYLENGTH_Bands', 'econstat_imputed_R'
]

# Check a sample of files across years
sample_files = [
    '0708_sr_gn_eul.tab', '0910_sr_gn_eul.tab', '1213_sr_gn_eul.tab',
    '1415_sr_gn_eul.tab', '1718_sr_gn_eul.tab', 'core_lettings_2018-19_eul.tab'
]

print(f'{"Column":<30}', end='')
for f in sample_files:
    print(f'{f[:10]:>12}', end='')
print()
print('-' * (30 + 12 * len(sample_files)))

for tc in target_cols:
    print(f'{tc:<30}', end='')
    for f in sample_files:
        df = spark.read.csv(f'{BRONZE}/{f}', header=True, inferSchema=False, sep='\t')
        present = '✓' if tc in df.columns else '✗'
        print(f'{present:>12}', end='')
    print()

Column                          0708_sr_gn  0910_sr_gn  1213_sr_gn  1415_sr_gn  1718_sr_gn  core_letti
------------------------------------------------------------------------------------------------------
YEAR                                     ✓           ✓           ✓           ✓

           ✓           ✓
GOVREG                                   ✓           ✓           ✓

           ✓           ✓           ✓
LETTYPE                                  ✓           ✓

           ✓           ✓           ✓           ✓
TENANCY                                  ✓

           ✓           ✓           ✓           ✓           ✓
HHMEMBT                       

...

           ✗           ✗
ETHNIC_Bands                             ✓           ✓

           ✓           ✓           ✓           ✓
TENANCYLENGTH_Bands                      ✗

           ✗           ✓           ✓           ✓           ✓
econstat_imputed_R            

           ✗           ✗           ✓           ✓           ✓

           ✗


## Summary of findings

Key observations that shaped `04_ingest_core.ipynb`:

- **42 files, 7 distinct schemas** (93–213 columns). Cannot glob — must read individually.
- **London filter**: `GOVREG == '7'` for files pre-2018, `GOVREG == 'E12000007'` for 2018–2022.
- **No borough breakdown** — CORE anonymises geography to region level only. Borough-level deprivation must come from a separate source (IMD).
- **Income/rent as string bands** — requires a midpoint UDF for any numeric analysis.
- **Blank strings in integer columns** — `HHMEMBT`, `BEDST`, `BED_MINUS_BEDSTANDARD` all need blank-to-null handling before casting.
- **`econstat_imputed_R`**, **`TENANCYLENGTH_Bands`**, and **`BED_MINUS_BEDSTANDARD`** only appear from 2012–13 onwards — older years get null for these columns.
- **`WTSHORTFALLHB_Bands`** (rent shortfall) is named `WTSHORTFALL_Bands` in some years.